In [1]:
import pandas as pd
df = pd.read_csv('../../02_Data/processed/final_merged_projects.csv')

c:\Users\seon\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\seon\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df.columns

Index(['projectID', 'currencySymbol', 'backersCount', 'phaseLabel',
       'enableBoardGameProperties', 'minPlayers', 'maxPlayers', 'minAge',
       'playTime', 'fundedInSeconds', 'projectTags', 'isDiscounted',
       'isFeatured', 'installmentMinPayment', 'hasLimitedStock',
       'productCanBePurchased', 'previous_campaigns_count', 'duration_days',
       'softclose', 'campaignGoal_usd_1m', 'fundsGathered_usd_1m',
       'price_usd_1m', 'campaignGoal_usd_6m', 'fundsGathered_usd_6m',
       'price_usd_6m', 'edge_density', 'saturation', 'brightness', 'contrast',
       'rewards_count', 'R1', 'G1', 'B1', 'R2', 'G2', 'B2', 'R3', 'G3', 'B3',
       'R4', 'G4', 'B4', 'ks_color_1', 'ks_color_2', 'ks_color_3',
       'ks_color_4', 'emotion_adjective_1', 'emotion_adjective_2',
       'emotion_adjective_3', 'emotion_adjective_4', 'creator_id_0',
       'creator_id_1', 'is_pledge_master_0', 'is_pledge_master_1',
       'is_backer_0', 'is_backer_1', 'is_prior_backer_0', 'is_prior_backer_1',
    

In [ ]:
drop_columns = [
    'projectID', 'backersCount', 'phaseLabel', 'isFeatured', 
    'installmentMinPayment', 'hasLimitedStock', 'productCanBePurchased', 
    'campaignGoal_usd_6m', 'fundsGathered_usd_6m', 'price_usd_6m',
    'is_backer_0', 'is_backer_1'  
]

df = df.drop(columns=drop_columns)

In [11]:
df['projectTags']

0      Exploration, Horror, Modern, Science Fiction, ...
1      History, Strategy, Wargame, Action, Collectibl...
2                             History, Strategy, Wargame
3                Card Game, Strategy, Party game, Family
4      Strategy, Resource management, Family, Worker ...
                             ...                        
478    Strategy, Wargame, Dice Game, Area Control, Co...
479                             Fantasy, Strategy, Humor
480        Dice Game, Multiplayer, Competitive, Campaign
481           Fantasy, Strategy, Terrain Building, Humor
482         Science Fiction, Area Control, Deck Building
Name: projectTags, Length: 483, dtype: object

In [13]:

# 2. FLAML 실행 코드
import numpy as np
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import re
target_col = 'fundsGathered_usd_1m'

q1 = df[target_col].quantile(0.25)
q3 = df[target_col].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df_filtered = df[(df[target_col] >= lower_bound) & (df[target_col] <= upper_bound)].copy()
df_filtered = df_filtered[df_filtered[target_col] > 0].reset_index(drop=True)

print(f"🧹 2단계: 1.5 IQR 타깃 정제 완료. (남은 행: {df_filtered.shape[0]}개)")

# ======================================================================
# 3. 새로운 X (독립변수 전체) 및 y (타깃 로그 변환) 분리 및 세팅
# ======================================================================
X_full = df_filtered.drop(columns=[target_col])
y_original = df_filtered[target_col]
y_log = np.log1p(y_original)


if 'projectTags' in df_filtered.columns:
    tags_dummies = df_filtered['projectTags'].str.get_dummies(sep=', ')
    # 기존 X_full에 쪼개진 태그 변수들을 옆으로 이어 붙이기
    X_full = pd.concat([X_full, tags_dummies], axis=1)

# LightGBM/XGBoost 특수문자 에러 방지 안전장치
X_full.columns = [re.sub(r'[ ,\{\}:"\]\[\-]', '_', col) for col in X_full.columns]

# 검증용 Train / Test 데이터셋 8:2 분할

# 1.5 IQR 정제된 데이터셋 분할 (기존 파일에서 구축된 X_full, y_log 그대로 사용)
X_train, X_test, y_train, y_test = train_test_split(X_full, y_log, test_size=0.2, random_state=42)

# AutoML 엔진 세팅
automl = AutoML()
automl_settings = {
    "time_budget": 60,         # 기계에게 탐색할 시간 60초 주기
    "metric": 'mae',
    "task": 'regression',
    "estimator_list": ['lgbm', 'xgboost'], # 우리가 주력으로 쓴 두 모델 타깃
    "seed": 42,
}

print("🤖 마이크로소프트 FLAML 초고속 AutoML 최적화 스타트...")
automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

# 예측 및 역변환
preds_real = np.expm1(automl.predict(X_test))
y_test_real = np.expm1(y_test)

print("\n👑 [AutoML 최종 결과 스코어]")
print("="*60)
print(f"🥇 선택된 최종 알고리즘: {automl.best_estimator}")
print(f"🎯 실전 검증 MAE 성적 : {mean_absolute_error(y_test_real, preds_real):,.2f} 달러")
print("="*60)

🧹 2단계: 1.5 IQR 타깃 정제 완료. (남은 행: 415개)
🤖 마이크로소프트 FLAML 초고속 AutoML 최적화 스타트...
[flaml.automl.logger: 06-13 20:25:03] {1752} INFO - task = regression
[flaml.automl.logger: 06-13 20:25:03] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 06-13 20:25:03] {1862} INFO - Minimizing error metric: mae
[flaml.automl.logger: 06-13 20:25:03] {1979} INFO - List of ML learners in AutoML Run: ['lgbm', 'xgboost']
[flaml.automl.logger: 06-13 20:25:03] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 06-13 20:25:04] {2417} INFO - Estimated sufficient time budget=1381s. Estimated necessary time budget=1s.
[flaml.automl.logger: 06-13 20:25:04] {2466} INFO -  at 0.2s,	estimator lgbm's best error=1.4616,	best estimator lgbm's best error=1.4616
[flaml.automl.logger: 06-13 20:25:04] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 06-13 20:25:04] {2466} INFO -  at 0.4s,	estimator lgbm's best error=1.4616,	best estimator lgbm's best error=1.4616
[flaml.automl.lo

In [17]:
import pandas as pd

# 1. FLAML이 최종 낙점한 LGBM 베스트 모델 객체 가져오기
best_lgbm_model = automl.model.estimator

# 2. 💡 [수정] X_train 대신, LightGBM 모델이 실제로 먹고 자란 진짜 변수 이름 리스트 추출
try:
    # LightGBM 내부에 저장된 피처 이름 가져오기
    feature_names = best_lgbm_model.feature_name_
except AttributeError:
    # 만약 에러가 날 경우 FLAML 객체가 기억하는 피처 이름으로 우회
    feature_names = automl.feature_names

# 3. 모델이 뱉은 중요도 배열 가져오기
importances = best_lgbm_model.feature_importances_

print(f"📐 [길이 대조 완료] 모델 내부 변수 개수: {len(feature_names)}개 | 중요도 배열 크기: {len(importances)}개")

# 4. 이제 완벽하게 일치하는 길이로 데이터프레임 빌드
df_automl_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

# 5. 중요도가 0보다 큰 '진짜 생존 변수'들만 필터링
automl_active_features = df_automl_imp[df_automl_imp['Importance'] > 0].reset_index(drop=True)

print("\n👑 [AutoML 최종 변수 생존자 리포트]")
print("=" * 65)
print(f"📊 1차 필터링 후 모델에 최종 투입된 변수: {len(feature_names)}개")
print(f"🔥 그 중 실제로 기여도가 발생한 진짜 변수: {len(automl_active_features)}개")
print("=" * 65)

print("\n🏆 [TOP 20] AutoML 모델을 캐리한 핵심 상위 변수 랭킹")
print("-" * 65)
print(automl_active_features.head(20))
print("-" * 65)

📐 [길이 대조 완료] 모델 내부 변수 개수: 140개 | 중요도 배열 크기: 140개

👑 [AutoML 최종 변수 생존자 리포트]
📊 1차 필터링 후 모델에 최종 투입된 변수: 140개
🔥 그 중 실제로 기여도가 발생한 진짜 변수: 69개

🏆 [TOP 20] AutoML 모델을 캐리한 핵심 상위 변수 랭킹
-----------------------------------------------------------------
                     Feature  Importance
0        campaignGoal_usd_1m          53
1            fundedInSeconds          39
2              duration_days          18
3                    likes_1          17
4                 saturation          15
5   Product_Question_count_1          15
6               price_usd_1m          15
7                   playTime          11
8               creator_id_1          11
9                    likes_0          11
10                        B4          10
11                maxPlayers           9
12                        G1           8
13                  contrast           8
14                        G2           8
15                    minAge           8
16               is_normal_1           7
17                   